In [ ]:
#  Query to get the features for users who were acquired on last 5 days, post removing the on purpose features

In [1]:
import pandas as pd
import numpy as np
import json
import lightgbm as lgb
from datetime import datetime, timezone

In [2]:
query="""

WITH
target_date AS (
    SELECT DATEADD(day, -6, CURRENT_DATE) AS d   -- their Day 5 = yesterday, fully closed out
),

-- ── trimmed to only what feeds the 20 features + dropoff-deviation calc ────
all_completed AS (
    SELECT
        j.ref_customer_id,
        j.customer_id,
        j.journey_id,

        (j.journey_created_at AT TIME ZONE 'UTC') AT TIME ZONE 'Asia/Dubai' AS journey_ts,
        DATE((j.journey_created_at AT TIME ZONE 'UTC') AT TIME ZONE 'Asia/Dubai') AS journey_dt,
        EXTRACT(HOUR FROM (j.journey_created_at AT TIME ZONE 'UTC') AT TIME ZONE 'Asia/Dubai') AS local_booking_hr,

        j.pickup_zone_name,
        j.drop_off_zone_name AS dropoff_zone_name,
        j.actual_discount_amount,

        j.estimate_drop_off_latitude,
        j.estimate_drop_off_longitude,
        j.actual_drop_off_latitude,
        j.actual_drop_off_longitude,
        j.actual_drop_off_time,
        j.estimate_drop_off_time,

        CASE
            WHEN j.estimate_drop_off_latitude  IS NOT NULL
             AND j.actual_drop_off_latitude     IS NOT NULL
            THEN
                2 * 6371 * ASIN(SQRT(
                    POWER(SIN(RADIANS(
                        (j.actual_drop_off_latitude - j.estimate_drop_off_latitude) / 2
                    )), 2)
                    + COS(RADIANS(j.estimate_drop_off_latitude))
                    * COS(RADIANS(j.actual_drop_off_latitude))
                    * POWER(SIN(RADIANS(
                        (j.actual_drop_off_longitude - j.estimate_drop_off_longitude) / 2
                    )), 2)
                ))
            ELSE NULL
        END                                                          AS dropoff_deviation_km,
        DATEDIFF(minute, j.estimate_drop_off_time, j.actual_drop_off_time) AS dropoff_time_diff

    FROM prod_etl_data.tbl_journey_master j
    WHERE j.journey_status IN (9, 10)
      AND j.journey_created_at IS NOT NULL
      AND journey_type = 1
)

,users AS (
    SELECT
        _id      AS ref_customer_id,
        useruid  AS customer_id,
        emailid,
        CASE WHEN countrycode LIKE '%+971%' THEN 1 ELSE 0 END AS is_uae_number
    FROM public.users
    WHERE usertype = 1
      AND status   = 1
)

-- ── STEP 3: Rank ALL trips, ALL history — unchanged, unrestricted scan ────
,trip_ranked AS (
    SELECT
        ac.*,
        ROW_NUMBER() OVER (PARTITION BY ac.ref_customer_id ORDER BY ac.journey_ts ASC) AS trip_rank
    FROM all_completed ac
    INNER JOIN users u ON ac.ref_customer_id = u.ref_customer_id
)

-- ── STEP 4: Cohort — true first trip = target_date only ────────────────────
,first_trips AS (
    SELECT
        ref_customer_id,
        customer_id,
        journey_dt              AS first_trip_dt,
        local_booking_hr        AS first_trip_hour,
        actual_discount_amount  AS first_trip_actual_discount_amount,
        dropoff_deviation_km    AS first_trip_dropoff_deviation_km,
        dropoff_time_diff       AS first_trip_dropoff_time_diff,
        pickup_zone_name        AS first_trip_pickup_zone,
        dropoff_zone_name       AS first_trip_dropoff_zone
    FROM trip_ranked
    WHERE trip_rank  = 1
      AND journey_dt = (SELECT d FROM target_date)
)

-- ── STEP 5: trips_day0_to_5 kept only to (a) drive completion_rate_day5
-- and (b) filter at-risk users at the end — not itself a feature ───────────
,day5_window AS (
    SELECT
        ft.ref_customer_id,
        COUNT(tr.journey_id) AS trips_day0_to_5
    FROM first_trips ft
    LEFT JOIN trip_ranked tr
           ON  tr.ref_customer_id = ft.ref_customer_id
          AND  tr.journey_dt     >= ft.first_trip_dt
          AND  tr.journey_dt     <= DATEADD(day, 5, ft.first_trip_dt)
    GROUP BY ft.ref_customer_id
)

-- ── STEP 6: cancellations — trimmed to customer_cancels_day5 +
-- total_requests_day5 (internal, for completion_rate_day5 only) ────────────
,cancellations AS (
    SELECT
        ft.ref_customer_id,
        COUNT(*)                                                AS total_requests_day5,
        SUM(CASE WHEN j.journey_status = 13 THEN 1 ELSE 0 END)  AS customer_cancels_day5
    FROM first_trips ft
    INNER JOIN prod_etl_data.tbl_journey_master j
            ON  j.ref_customer_id = ft.ref_customer_id
           AND  DATE((j.journey_created_at AT TIME ZONE 'UTC') AT TIME ZONE 'Asia/Dubai') >= ft.first_trip_dt
           AND  DATE((j.journey_created_at AT TIME ZONE 'UTC') AT TIME ZONE 'Asia/Dubai') <= DATEADD(day, 5, ft.first_trip_dt)
    GROUP BY ft.ref_customer_id
)

,email_signals AS (
    SELECT
        ft.ref_customer_id,
        COUNT(DISTINCT CASE WHEN u_all.emailid = u_target.emailid
                             AND u_target.emailid IS NOT NULL AND u_target.emailid != ''
                        THEN u_all._id END) AS accounts_per_email
    FROM first_trips ft
    INNER JOIN users u_target ON ft.ref_customer_id = u_target.ref_customer_id
    LEFT JOIN public.users u_all
           ON u_all.emailid = u_target.emailid
          AND u_target.emailid IS NOT NULL
          AND u_target.emailid != ''
          AND u_all.usertype = 1
    GROUP BY ft.ref_customer_id
)

-- ── Amplitude session events — trimmed to only the event types that feed
-- funnel depth + the specific post-trip features needed ───────────────────
,amp_base AS (
    SELECT ft.customer_id, e.session_id,
           (e.event_time::timestamp AT TIME ZONE 'UTC') AT TIME ZONE 'Asia/Dubai' AS event_ts,
           DATE((e.event_time::timestamp AT TIME ZONE 'UTC') AT TIME ZONE 'Asia/Dubai') AS event_dt,
           LOWER(TRIM(e.event_type)) AS event_type,
           e.os_name
    FROM amplitude_customer_app.events e
    INNER JOIN first_trips ft ON ft.customer_id = e.user_id
           AND DATE((e.event_time::timestamp AT TIME ZONE 'UTC') AT TIME ZONE 'Asia/Dubai')
               BETWEEN ft.first_trip_dt AND DATEADD(day, 5, ft.first_trip_dt)
    WHERE e.session_id <> -1 AND e.session_id IS NOT NULL AND e.user_id IS NOT NULL

    UNION ALL

    SELECT ft.customer_id, e.session_id,
           (e.event_time::timestamp AT TIME ZONE 'UTC') AT TIME ZONE 'Asia/Dubai' AS event_ts,
           DATE((e.event_time::timestamp AT TIME ZONE 'UTC') AT TIME ZONE 'Asia/Dubai') AS event_dt,
           LOWER(TRIM(e.event_type)) AS event_type,
           e.os_name
    FROM amplitude_customer_app.events e
    INNER JOIN first_trips ft ON ft.ref_customer_id = e.user_id
           AND DATE((e.event_time::timestamp AT TIME ZONE 'UTC') AT TIME ZONE 'Asia/Dubai')
               BETWEEN ft.first_trip_dt AND DATEADD(day, 5, ft.first_trip_dt)
    WHERE e.session_id <> -1 AND e.session_id IS NOT NULL AND e.user_id IS NOT NULL AND e.user_id NOT LIKE 'CUS_%'
)

,amp_sessions AS (
    SELECT
        customer_id,
        session_id,
        MIN(event_dt) AS session_dt,
        DATEDIFF(second, MIN(event_ts), MAX(event_ts)) / 60.0 AS session_duration_mins,
        -- had_page_view/had_journey_intent kept only as inputs to max_funnel_depth
        -- below, not selected as their own feature anywhere downstream
        MAX(CASE
            WHEN event_type LIKE '%book_journey_clicked%' THEN 4
            WHEN event_type LIKE '%vehiclecategory%' OR event_type LIKE '%vehicle_detail%' THEN 3
            WHEN event_type LIKE '%journeyintent_started%' THEN 2
            WHEN event_type LIKE '%page_view%' THEN 1
            ELSE 0 END) AS max_funnel_depth,
        MAX(CASE WHEN event_type LIKE '%vehiclecategory%' OR event_type LIKE '%vehicle_category%' THEN 1 ELSE 0 END) AS had_vehicle_cat,
        MAX(CASE WHEN event_type LIKE '%book_journey_clicked%' THEN 1 ELSE 0 END) AS had_book_click,
        COUNT(CASE WHEN event_type LIKE '%location_search%' THEN 1 END) AS location_searches,
        MAX(os_name) AS os_name
    FROM amp_base
    GROUP BY 1, 2
)

,session_features AS (
    SELECT
        ft.customer_id,
        COUNT(DISTINCT CASE WHEN s.session_dt = ft.first_trip_dt THEN s.session_id END) AS sessions_on_day0,
        COUNT(DISTINCT CASE WHEN s.session_dt > ft.first_trip_dt THEN s.session_id END) AS sessions_after_first_trip,
        AVG(CASE WHEN s.session_dt = ft.first_trip_dt THEN s.session_duration_mins END) AS avg_session_dur_day0,
        AVG(CASE WHEN s.session_dt > ft.first_trip_dt THEN s.session_duration_mins END) AS avg_session_dur_after_trip,
        MAX(CASE WHEN s.session_dt > ft.first_trip_dt THEN s.max_funnel_depth END)      AS max_funnel_depth_after_trip,
        MAX(CASE WHEN s.session_dt > ft.first_trip_dt THEN s.had_vehicle_cat END)       AS post_trip_reached_vehicle_cat,
        MAX(CASE WHEN s.session_dt > ft.first_trip_dt THEN s.had_book_click END)        AS post_trip_clicked_book,
        SUM(CASE WHEN s.session_dt > ft.first_trip_dt
                  AND s.had_book_click = 1 THEN 1 ELSE 0 END)                          AS post_trip_booking_sessions,
        SUM(s.location_searches)                                                       AS total_location_searches_day5,
        MAX(s.os_name)                                                                  AS app_platform
    FROM first_trips ft
    LEFT JOIN amp_sessions s ON s.customer_id = ft.customer_id
    GROUP BY ft.customer_id, ft.first_trip_dt
)

-- ── FINAL: one row per user, exactly the 20 features + identifiers ────────
SELECT
    ft.ref_customer_id,
    ft.customer_id,

    ft.first_trip_hour,
    ft.first_trip_pickup_zone,
    ft.first_trip_dropoff_zone,
    ft.first_trip_actual_discount_amount,
    ft.first_trip_dropoff_deviation_km,
    ft.first_trip_dropoff_time_diff,

    u.is_uae_number,

    c.customer_cancels_day5,
    CASE WHEN c.total_requests_day5 > 0
         THEN d5.trips_day0_to_5 * 1.0 / c.total_requests_day5
         ELSE 1.0 END                                    AS completion_rate_day5,

    es.accounts_per_email,

    sf.sessions_on_day0,
    sf.sessions_after_first_trip,
    sf.avg_session_dur_day0,
    sf.avg_session_dur_after_trip,
    sf.max_funnel_depth_after_trip,
    sf.post_trip_reached_vehicle_cat,
    sf.post_trip_clicked_book,
    sf.post_trip_booking_sessions,
    sf.total_location_searches_day5,
    sf.app_platform

FROM first_trips ft
LEFT JOIN day5_window     d5 ON ft.ref_customer_id = d5.ref_customer_id
LEFT JOIN cancellations    c ON ft.ref_customer_id = c.ref_customer_id
LEFT JOIN email_signals   es ON ft.ref_customer_id = es.ref_customer_id
LEFT JOIN users            u ON ft.ref_customer_id = u.ref_customer_id
LEFT JOIN session_features sf ON ft.customer_id = sf.customer_id

WHERE d5.trips_day0_to_5 = 1   -- at-risk filter, applied at source
ORDER BY 1, 2


"""

In [3]:
import psycopg2
redshift_host = "zed-production-redshift-cluster-1.ccgbflzbwwtj.ap-south-2.redshift.amazonaws.com"
redshift_port = 5439
database = "dev"
db_user = "awsuser"
db_password = "w%J>5&HUj>iB$zF["


conn = psycopg2.connect(
    host=redshift_host,
    port=redshift_port,
    database=database,
    user=db_user,
    password=db_password
)

df_score = pd.read_sql_query(query, conn)
conn.close()
 

/var/folders/g2/1_yz974j2ggcl7nwk6g_45w5w85bvx/T/ipykernel_68854/3136447684.py:17: UserWarning: pandas only supports SQLAlchemy connectable (engine/connection) or database string URI or sqlite3 DBAPI2 connection. Other DBAPI2 objects are not tested. Please consider using SQLAlchemy.
  df_score = pd.read_sql_query(query, conn)


In [4]:
df_score.shape

(426, 22)

In [5]:

# ── 1. LOAD SAVED MODEL + FEATURE METADATA ──────────────────────────────────
model = lgb.Booster(model_file='churn_model_top20_final.txt')
meta = json.load(open('churn_model_top20_features.json'))
feature_list = meta['feature_list']
cat_cols = meta['cat_cols']

In [6]:

# ── 2. SPLIT OFF IDENTIFIERS — never feed these to the model ────────────────
ids = df_score[['ref_customer_id', 'customer_id']].copy()
X = df_score.drop(columns=['ref_customer_id', 'customer_id']).copy()



In [7]:

# ── 3. CATEGORICAL COLUMNS — same treatment as training ─────────────────────
CAT_COLS = ['first_trip_pickup_zone', 'first_trip_dropoff_zone', 'app_platform']
for col in CAT_COLS:
    X[col] = X[col].fillna('__missing__').astype(str).astype('category')


# ── 4. FILL-ZERO NUMERIC — null means "no activity", same as training ───────
FILL_ZERO_NUMERIC = [
    'customer_cancels_day5', 'sessions_on_day0', 'sessions_after_first_trip',
    'post_trip_booking_sessions', 'total_location_searches_day5',
    'max_funnel_depth_after_trip',
]
for col in FILL_ZERO_NUMERIC:
    X[col] = pd.to_numeric(X[col], errors='coerce').fillna(0)


# ── 5. BINARY-STYLE COLUMNS — fill null with 0, cast int ────────────────────
BINARY_COLS = ['is_uae_number', 'post_trip_reached_vehicle_cat', 'post_trip_clicked_book']
for col in BINARY_COLS:
    X[col] = pd.to_numeric(X[col], errors='coerce').fillna(0).astype(int)

# ── 6. LEAVE-AS-NaN — meaningful null, LightGBM handles natively ────────────
LEAVE_NAN = [
    'avg_session_dur_day0', 'avg_session_dur_after_trip',
    'first_trip_dropoff_deviation_km', 'first_trip_dropoff_time_diff',
]
for col in LEAVE_NAN:
    X[col] = pd.to_numeric(X[col], errors='coerce')


# ── 7. REMAINING NUMERIC — straightforward, coerce and fill sensibly ────────

X['first_trip_actual_discount_amount'] = pd.to_numeric( X['first_trip_actual_discount_amount'], errors='coerce').fillna(0)
X['first_trip_hour'] = pd.to_numeric(X['first_trip_hour'], errors='coerce')
X['completion_rate_day5'] = pd.to_numeric(X['completion_rate_day5'], errors='coerce')


In [8]:

# ── 8. VERIFY column order/set matches exactly what the model was trained on ─
assert set(X.columns) == set(feature_list), (
    f"Column mismatch!\nIn X not in feature_list: {set(X.columns) - set(feature_list)}\n"
    f"In feature_list not in X: {set(feature_list) - set(X.columns)}"
)
X = X[feature_list]   # enforce exact training-time column order


In [9]:

# ── 9. SCORE ─────────────────────────────────────────────────────────────────
scores = model.predict(X)


In [10]:

# ── 10. SEGMENT — cutoffs from the validated OOT threshold analysis ─────────
def assign_segment(score):
    if score >= 0.8:
        return 'High'
    elif score >= 0.6:
        return 'Medium'
    return 'Low'

output = ids.copy()
output['churn_risk_score'] = scores.round(4)
output['churn_risk_segment'] = [assign_segment(s) for s in scores]
output['scored_at'] = datetime.now(timezone.utc).isoformat()

print(f"Scored {len(output)} users")
print(output['churn_risk_segment'].value_counts())
print(output.head())



Scored 426 users
churn_risk_segment
Medium    226
High      139
Low        61
Name: count, dtype: int64
            ref_customer_id     customer_id  churn_risk_score  \
0  66814244ebc4992d6b97e7ba  CUS_JCAFU52254            0.9477   
1  66b8d47e8298ed34d1b3e008  CUS_JGIKZ12475            0.6635   
2  6756d0a471b083bbe19e5822  CUS_LOWKK84073            0.8666   
3  677a4cdd04cbf11302849226  CUS_WIPMF37759            0.9208   
4  677e243d3fd8dd6cb127d031  CUS_PSHMN09862            0.5405   

  churn_risk_segment                         scored_at  
0               High  2026-07-23T11:28:58.427599+00:00  
1             Medium  2026-07-23T11:28:58.427599+00:00  
2               High  2026-07-23T11:28:58.427599+00:00  
3               High  2026-07-23T11:28:58.427599+00:00  
4                Low  2026-07-23T11:28:58.427599+00:00  


In [11]:
output

,ref_customer_id,customer_id,churn_risk_score,churn_risk_segment,scored_at
0,66814244ebc4992d6b97e7ba,CUS_JCAFU52254,0.9477,High,2026-07-23T11:28:58.427599+00:00
1,66b8d47e8298ed34d1b3e008,CUS_JGIKZ12475,0.6635,Medium,2026-07-23T11:28:58.427599+00:00
2,6756d0a471b083bbe19e5822,CUS_LOWKK84073,0.8666,High,2026-07-23T11:28:58.427599+00:00
3,677a4cdd04cbf11302849226,CUS_WIPMF37759,0.9208,High,2026-07-23T11:28:58.427599+00:00
4,677e243d3fd8dd6cb127d031,CUS_PSHMN09862,0.5405,Low,2026-07-23T11:28:58.427599+00:00
...,...,...,...,...,...
421,6a5a835f559206241b1bb498,CUS_UWATP39262,0.9322,High,2026-07-23T11:28:58.427599+00:00
422,6a5a83621cde5e18bfb140f3,CUS_ISUEE25876,0.8038,High,2026-07-23T11:28:58.427599+00:00
423,6a5a83658cef017f0ce7ce19,CUS_YWEHN15250,0.7229,Medium,2026-07-23T11:28:58.427599+00:00
424,6a5a83718cef017f0ce7ce6e,CUS_NPLMP17999,0.8517,High,2026-07-23T11:28:58.427599+00:00


#  update the .py file from here

#  randomly sample for A/ B testing

In [14]:
import hashlib
import os, requests

In [26]:
import requests
print(requests.__version__)

2.32.4


In [15]:
def stable_hash(value: str) -> int:
    """Deterministic integer hash — independent of Python's randomized hash
    seed (which changes between process runs) and independent of row order
    in the dataframe. Same customer_id always produces the same number."""
    return int(hashlib.md5(str(value).encode()).hexdigest(), 16)


In [16]:
def assign_experiment_group(
    output: pd.DataFrame,
    id_col: str = 'customer_id',
    segment_col: str = 'churn_risk_segment',
) -> pd.DataFrame:
    """
    Splits each day's scored cohort into Test/Control, exactly 50/50 (±1 for
    odd counts) WITHIN each risk segment independently — so High, Medium,
    and Low each get their own balanced split, not one split across the
    whole day's list.

    Deterministic per user: reruns of the same day's job, or the SQL query
    returning rows in a different order, always produce the identical
    assignment for a given customer_id. Handles any daily volume and any
    per-segment count automatically, since the split is recomputed fresh
    from whoever's actually in that day's dataframe.
    """
    df = output.copy()
    df['_hash'] = df[id_col].map(stable_hash)
    def split_segment(g):
        g = g.sort_values('_hash').reset_index(drop=True)
        n_test = len(g) // 2                     # floor; leftover (odd count) goes to Control
        g['experiment_group'] = ['Test'] * n_test + ['Control'] * (len(g) - n_test)
        return g

    df = df.groupby(segment_col, group_keys=False).apply(split_segment)
    return df.drop(columns='_hash')

output = assign_experiment_group(output)

print(output.groupby(['churn_risk_segment', 'experiment_group']).size().unstack())

experiment_group    Control  Test
churn_risk_segment               
High                     70    69
Low                      31    30
Medium                  113   113


/var/folders/g2/1_yz974j2ggcl7nwk6g_45w5w85bvx/T/ipykernel_68854/3149148522.py:26: FutureWarning: DataFrameGroupBy.apply operated on the grouping columns. This behavior is deprecated, and in a future version of pandas the grouping columns will be excluded from the operation. Either pass `include_groups=False` to exclude the groupings or explicitly select the grouping columns after groupby to silence this warning.
  df = df.groupby(segment_col, group_keys=False).apply(split_segment)


In [19]:
output.head()

,ref_customer_id,customer_id,churn_risk_score,churn_risk_segment,scored_at,experiment_group
0,6a463fc18e5fcea55cc725e6,CUS_ROFRT68523,0.8870,High,2026-07-23T11:28:58.427599+00:00,Test
1,69ae5edc68b7cf3d2182d736,CUS_WGFZC48637,0.9284,High,2026-07-23T11:28:58.427599+00:00,Test
2,6a58a3618cef017f0cd649fc,CUS_SBMDC79721,0.9137,High,2026-07-23T11:28:58.427599+00:00,Test
3,6a57ca368cef017f0cd01929,CUS_CZYJP14173,0.8716,High,2026-07-23T11:28:58.427599+00:00,Test
4,6a591da7559206241b0f4c29,CUS_YFLZN00436,0.9284,High,2026-07-23T11:28:58.427599+00:00,Test


In [18]:
filename = f"output_{datetime.now().strftime('%Y%m%d')}.csv"
output[output['experiment_group'] == 'Test'].to_csv(filename, index=False)

In [17]:
pwd

'/Users/richa.kumari/Documents/Churn_Project_June_30'

#  SSO enable and clevertap connection

In [20]:
output_test=output[output['experiment_group'] == 'Test']
output_test.shape


(212, 6)

In [3]:
import pandas as pd
from datetime import datetime

today = datetime.now().strftime("%Y-%m-%d")

data = [
    {"customer_id": "CUS_MGYWI99040", "churn_risk_segment": "High",   "scored_at": today},
    {"customer_id": "CUS_LFMSI05028", "churn_risk_segment": "High",   "scored_at": today},
    {"customer_id": "CUS_WCMTM62255", "churn_risk_segment": "Medium", "scored_at": today},
    {"customer_id": "CUS_EOFSH31378",   "churn_risk_segment": "Medium", "scored_at": today},
    {"customer_id": "CUS_JEURQ64765", "churn_risk_segment": "Low",    "scored_at": today},
    {"customer_id": "CUS_DPLF4374", "churn_risk_segment": "Low",    "scored_at": today},
]

output = pd.DataFrame(data)

In [6]:
# change this to output_test later
records = []
for _, row in output_test.iterrows():
    records.append({
        "identity": str(row["customer_id"]),
        "type": "profile",
        "profileData": {
            "Churn Risk Segment": row["churn_risk_segment"],
            "Churn Scored At": str(row["scored_at"])
        }
    })

In [7]:
records

[{'identity': 'CUS_MGYWI99040',
  'type': 'profile',
  'profileData': {'Churn Risk Segment': 'High',
   'Churn Scored At': '2026-07-24'}},
 {'identity': 'CUS_LFMSI05028',
  'type': 'profile',
  'profileData': {'Churn Risk Segment': 'High',
   'Churn Scored At': '2026-07-24'}},
 {'identity': 'CUS_WCMTM62255',
  'type': 'profile',
  'profileData': {'Churn Risk Segment': 'Medium',
   'Churn Scored At': '2026-07-24'}},
 {'identity': 'CUS_EOFSH31378',
  'type': 'profile',
  'profileData': {'Churn Risk Segment': 'Medium',
   'Churn Scored At': '2026-07-24'}},
 {'identity': 'CUS_JEURQ64765',
  'type': 'profile',
  'profileData': {'Churn Risk Segment': 'Low',
   'Churn Scored At': '2026-07-24'}},
 {'identity': 'CUS_DPLF4374',
  'type': 'profile',
  'profileData': {'Churn Risk Segment': 'Low',
   'Churn Scored At': '2026-07-24'}}]

In [10]:
import os
import requests
os.environ["CLEVERTAP_ACCOUNT_ID"] = "RK8-4W9-RZ7Z"  # staging environment      
os.environ["CLEVERTAP_PASSCODE"] = "EHE-AKV-MHEL"  # staging environment      

# os.environ["CLEVERTAP_ACCOUNT_ID"] = "R57-8K5-476Z"   # prod environment    
# os.environ["CLEVERTAP_PASSCODE"] = "ETQ-KWZ-ATUL"  

ACCOUNT_ID = os.environ["CLEVERTAP_ACCOUNT_ID"]
PASSCODE   = os.environ["CLEVERTAP_PASSCODE"]

URL        = "https://api.clevertap.com/1/upload"   

session = requests.Session()
session.headers.update({
    "X-CleverTap-Account-Id": ACCOUNT_ID,
    "X-CleverTap-Passcode": PASSCODE,
    "Content-Type": "application/json",
})
session.verify = os.environ.get("REQUESTS_CA_BUNDLE", True)

BATCH_SIZE = 1000
failed = []

for i in range(0, len(records), BATCH_SIZE):
    batch = records[i:i+BATCH_SIZE]
    try:
        r = session.post(URL, json={"d": batch}, timeout=60)
        body = r.json()
    except Exception as e:
        failed.append((i, repr(e)))
        continue

    if r.status_code != 200 or body.get("status") != "success":
        failed.append((i, body))

    print(f"Batch {i//BATCH_SIZE + 1}: {r.status_code} | "
          f"processed={body.get('processed')} unprocessed={len(body.get('unprocessed', []))}")

print("Failed batches:", failed)

Batch 1: 200 | processed=6 unprocessed=0
Failed batches: []


#  write the table in redshift and update it with append it daily


#  Write to redshift

In [25]:
import psycopg2
import pandas as pd
from psycopg2.extras import execute_values

SCHEMA, TABLE = "prod_etl_temp", "new_user_churn_scores"

conn = psycopg2.connect(
    host="zed-production-redshift-cluster-1.ccgbflzbwwtj.ap-south-2.redshift.amazonaws.com",
    port=5439, database="dev", user="awsuser", password= "w%J>5&HUj>iB$zF[",
)
df = output.copy()
df["ingested_at"] = datetime.now(timezone.utc).replace(tzinfo=None)

cols = ", ".join(f'"{c}"' for c in df.columns)
rows = list(df.astype(object).where(pd.notnull(df), None).itertuples(index=False, name=None))

with conn, conn.cursor() as cur:
    cur.execute(f"""
        CREATE TABLE IF NOT EXISTS "{SCHEMA}"."{TABLE}" (
            ref_customer_id     VARCHAR(64),
            customer_id         VARCHAR(64),
            churn_risk_score    DOUBLE PRECISION,
            churn_risk_segment  VARCHAR(32),
            scored_at           VARCHAR(32),
            experiment_group    VARCHAR(16),
            ingested_at         TIMESTAMP
        )
        DISTSTYLE KEY DISTKEY(customer_id)
        SORTKEY(ingested_at);
    """)
    cur.execute(
        f'DELETE FROM "{SCHEMA}"."{TABLE}" WHERE ingested_at::date = current_date;'
    )
    execute_values(
        cur, f'INSERT INTO "{SCHEMA}"."{TABLE}" ({cols}) VALUES %s', rows, page_size=5000
    )

conn.close()
print(f"Appended {len(df):,} rows")

Appended 426 rows
